# 08 · Build the analysis JSON (the "contract" between team members)

This notebook produces the **full analysis result** for each dataset as one JSON file, in the format the API will send to the dashboard:

- `contracts/example_incidents.json`
- `contracts/example_sales.json`

Every number in them is computed from your real data. They're useful in three ways:
- **Business analysts** can see exactly what the dashboard will show.
- **The frontend person** can build every screen from them right now.
- **The backend person** knows exactly what the API must return.

The meaning of every field is explained in `contracts/CONTRACT.md`.

Sections in each file:

| Section | Answers the analyst's question |
|---|---|
| `meta` | What data is this, which period, which model? |
| `data_quality` | Can I trust the data? What was removed and why? |
| `kpis` | How are we doing now vs before? |
| `trends` | How did it change week by week? |
| `breakdowns` | Which team / category / region is worst? |
| `root_causes` | *Why* do problems happen? |
| `anomalies` | Which weeks were unusual? |
| `forecast` | How much work is coming next week? |
| `predictions` | Which open items are at risk right now, and why? |
| `recommendations` | What should we do, and what will it gain? |
| `impact_simulation` | Before vs after, if we follow the recommendations |
| `model_performance` | How good is the prediction model? |

In [1]:
from pathlib import Path
import json
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
INTERIM, PROCESSED, REPORTS = ROOT / "data/interim", ROOT / "data/processed", ROOT / "reports"
CONTRACTS = ROOT / "contracts"
CONTRACTS.mkdir(exist_ok=True)
GENERATED_AT = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

## Shared helpers
- `num()` rounds numbers and turns numpy numbers into plain Python numbers (JSON can't store numpy types).
- `kpi()` builds one KPI card: value now, value before, change, and a status (good / warning / critical).
- `train_model()` uses the same cautious LightGBM settings as notebook 07.
- `explain()` uses LightGBM's built-in **SHAP values** (`pred_contrib=True`) to find the top reasons behind each prediction.

In [2]:
def num(x, d=3):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    if isinstance(x, (np.integer, int)):
        return int(x)
    return round(float(x), d)

def pct_change(now, before):
    return None if not before else num((now - before) / before * 100, 1)

def kpi(id, label, now, before, unit, definition, higher_is_better, warn=None, crit=None, d=3, mode="level"):
    """Status colour rules:
    - mode="level": compare the current value to warning/critical thresholds
    - mode="change": compare the % change vs previous period (warn/crit = % drop or rise that is bad)
    - higher_is_better=None: a volume metric that is neither good nor bad -> status "info"."""
    change = None if before is None else now - before
    chg_pct = pct_change(now, before)
    if higher_is_better is None:
        status, trend = "info", (None if change is None else "up" if change > 0 else "down" if change < 0 else "flat")
    else:
        if mode == "change":
            bad = -chg_pct if higher_is_better else chg_pct      # how many % it moved in the bad direction
            status = "critical" if bad >= crit else "warning" if bad >= warn else "good"
        elif higher_is_better:
            status = "critical" if now <= crit else "warning" if now <= warn else "good"
        else:
            status = "critical" if now >= crit else "warning" if now >= warn else "good"
        trend = None if change is None else ("flat" if change == 0 else
                                             "improving" if (change > 0) == higher_is_better else "worsening")
    return {"id": id, "label": label, "value": num(now, d), "unit": unit,
            "previous_value": num(before, d), "change": num(change, d), "change_pct": chg_pct,
            "trend": trend, "higher_is_better": higher_is_better, "status": status,
            "status_rule": None if higher_is_better is None else
                           {"mode": mode, "warning": warn, "critical": crit,
                            "meaning": "thresholds on % change in the bad direction" if mode == "change" else "thresholds on the value"},
            "definition": definition}

PARAMS = dict(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=200, reg_lambda=5,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)

def train_model(X, y, is_train):
    m = lgb.LGBMClassifier(**PARAMS).fit(X[is_train], y[is_train])
    p = m.predict_proba(X)[:, 1]
    yt, pt = y[~is_train].values, p[~is_train.values]
    top10 = pt >= np.quantile(pt, 0.9)
    perf = {"algorithm": "LightGBM (gradient boosted trees)", "test_auc": num(roc_auc_score(yt, pt)),
            "train_auc": num(roc_auc_score(y[is_train], p[is_train.values])),
            "base_rate_test": num(yt.mean()), "precision_top_10pct": num(yt[top10].mean()),
            "lift_top_10pct": num(yt[top10].mean() / yt.mean(), 2), "n_features": X.shape[1],
            "train_rows": int(is_train.sum()), "test_rows": int((~is_train).sum())}
    return m, p, perf

def explain(model, X_rows, X_train, labels, k=3):
    """Top-k reasons per row from SHAP values (log-odds contributions)."""
    contrib = model.predict(X_rows, pred_contrib=True)
    cols = list(X_rows.columns)
    out = []
    for i in range(len(X_rows)):
        c = contrib[i, :-1]
        order = np.argsort(-np.abs(c))[:k]
        reasons = []
        for j in order:
            f = cols[j]
            v = X_rows.iloc[i, j]
            if isinstance(X_train[f].dtype, pd.CategoricalDtype):
                v_out, typ_out = str(v), str(X_train[f].mode().iloc[0])
                typ_word, v_txt, typ_txt = "most common", v_out, typ_out
            else:
                v_out, typ_out = num(v, 2), num(X_train[f].median(), 2)
                typ_word, v_txt, typ_txt = "usual", f"{v:,.2f}".rstrip("0").rstrip("."), f"{X_train[f].median():,.2f}".rstrip("0").rstrip(".")
            direction = "increases risk" if c[j] > 0 else "decreases risk"
            reasons.append({"feature": f, "label": labels.get(f, f), "value": v_out, "typical_value": typ_out,
                            "effect": num(c[j], 3), "direction": direction,
                            "sentence": f"{labels.get(f, f)}: {v_txt} ({typ_word}: {typ_txt}), {direction}"})
        out.append(reasons)
    return out

def risk_level(p, hi, med):
    return "high" if p >= hi else "medium" if p >= med else "low"

def dq_section(report_csv, rows_in, rows_used, item_name, warnings, modules):
    rep = pd.read_csv(report_csv)
    steps = [{"step": r.step, "rows_before": int(r.rows_before), "rows_after": int(r.rows_after),
              "rows_removed": int(r.rows_removed), "note": "" if pd.isna(r.note) else r.note} for r in rep.itertuples()]
    return {"rows_in_raw_file": rows_in, f"{item_name}_used_for_analysis": rows_used,
            "cleaning_steps": steps, "warnings": warnings, "modules": modules}

---
# Part 1: Incidents

In [3]:
inc = pd.read_parquet(INTERIM / "incidents_clean.parquet")
feat = pd.read_parquet(PROCESSED / "incidents_features.parquet")
inc = inc.merge(feat[["number", "split"]], on="number")

cutoff = feat.loc[feat["split"] == "test", "opened_at"].min()
period_len = feat["opened_at"].max() - cutoff
prev_start = cutoff - period_len
cur = inc[inc["opened_at"] >= cutoff]
prev = inc[(inc["opened_at"] >= prev_start) & (inc["opened_at"] < cutoff)]
print("Current period :", cur["opened_at"].min(), "→", cur["opened_at"].max(), len(cur), "tickets")
print("Previous period:", prev["opened_at"].min(), "→", prev["opened_at"].max(), len(prev), "tickets")

Current period : 2016-05-10 10:02:00 → 2016-05-31 20:55:00 4929 tickets
Previous period: 2016-04-18 23:38:00 → 2016-05-10 10:01:00 5524 tickets


### KPIs (current period vs previous period of equal length)

In [4]:
def inc_kpis(df):
    return {"tickets": len(df), "breach": df["breached"].mean(), "median_h": df["hours_to_resolve"].median(),
            "reassigned": (df["final_reassignment_count"] > 0).mean(), "reopened": (df["final_reopen_count"] > 0).mean(),
            "high_prio": (df["priority_n"] <= 2).mean()}
k_now, k_prev = inc_kpis(cur), inc_kpis(prev)

inc_kpi_list = [
    kpi("ticket_volume", "Tickets received", k_now["tickets"], k_prev["tickets"], "tickets",
        "Number of tickets opened in the period", higher_is_better=None, d=0),
    kpi("sla_breach_rate", "SLA breach rate", k_now["breach"], k_prev["breach"], "ratio",
        "Share of tickets that missed their SLA deadline", False, 0.25, 0.35),
    kpi("median_resolution_hours", "Median time to resolve", k_now["median_h"], k_prev["median_h"], "hours",
        "Median hours from opening to resolution", False, 24, 48, d=1),
    kpi("reassignment_rate", "Tickets reassigned at least once", k_now["reassigned"], k_prev["reassigned"], "ratio",
        "Share of tickets passed to another team at least once", False, 0.35, 0.50),
    kpi("reopen_rate", "Reopened tickets", k_now["reopened"], k_prev["reopened"], "ratio",
        "Share of tickets reopened after being resolved", False, 0.02, 0.05),
    kpi("high_priority_share", "Critical + High priority share", k_now["high_prio"], k_prev["high_prio"], "ratio",
        "Share of tickets with priority 1 or 2", False, 0.05, 0.10),
]
pd.DataFrame(inc_kpi_list)[["label", "value", "previous_value", "change_pct", "trend", "status"]]

,label,value,previous_value,change_pct,trend,status
0,Tickets received,4929.000,5524.000,-10.8,down,info
1,SLA breach rate,0.225,0.234,-4.1,improving,good
2,Median time to resolve,2.900,2.100,34.9,worsening,good
3,Tickets reassigned at least once,0.368,0.396,-7.0,improving,warning
4,Reopened tickets,0.008,0.008,2.1,worsening,good
5,Critical + High priority share,0.019,0.028,-30.1,improving,good


### Trends, breakdowns, root causes, anomalies

In [5]:
w = inc.set_index("opened_at").resample("W-SUN")
weekly = pd.DataFrame({"tickets": w.size(), "breach_rate": w["breached"].mean(),
                       "median_resolution_hours": w["hours_to_resolve"].median()}).dropna()
inc_trends = {"granularity": "week", "series": [
    {"week_ending": d.strftime("%Y-%m-%d"), "tickets": int(r.tickets), "breach_rate": num(r.breach_rate),
     "median_resolution_hours": num(r.median_resolution_hours, 1)} for d, r in weekly.iterrows()]}

def breakdown(df, col, top, label, min_n=50):
    total_breaches = df["breached"].sum()
    g = df.groupby(col).agg(tickets=("number", "size"), breach_rate=("breached", "mean"),
                            breaches=("breached", "sum"), median_resolution_hours=("hours_to_resolve", "median"),
                            avg_reassignments=("final_reassignment_count", "mean"))
    g = g[g["tickets"] >= min_n].sort_values("breaches", ascending=False).head(top)
    overall = df["breached"].mean()
    return {"dimension": col, "label": label, "sorted_by": "breaches", "rows": [
        {"value": str(i), "tickets": int(r.tickets), "breach_rate": num(r.breach_rate), "breaches": int(r.breaches),
         "share_of_all_breaches": num(r.breaches / total_breaches), "vs_overall_breach_rate": num(r.breach_rate - overall),
         "median_resolution_hours": num(r.median_resolution_hours, 1), "avg_reassignments": num(r.avg_reassignments, 2),
         "flag": "above_average" if r.breach_rate > overall * 1.2 else "normal"} for i, r in g.iterrows()]}

inc_breakdowns = [breakdown(cur, "assignment_group", 8, "Team (assignment group)"),
                  breakdown(cur, "category", 8, "Category"),
                  breakdown(cur, "priority_n", 4, "Priority (1 = Critical)", min_n=1)]

# Root cause 1: reassignment
r0 = inc[inc["final_reassignment_count"] == 0]["breached"].mean()
r1 = inc[inc["final_reassignment_count"] == 1]["breached"].mean()
r2 = inc[inc["final_reassignment_count"] >= 2]
r5 = inc[inc["final_reassignment_count"] >= 5]["breached"].mean()
cur_r2 = cur[cur["final_reassignment_count"] >= 2]
# Root cause 2: reopened
ro1 = inc[inc["final_reopen_count"] > 0]["breached"].mean()
ro0 = inc[inc["final_reopen_count"] == 0]["breached"].mean()
# Root cause 3: teams with high breach in current period
team = breakdown(cur, "assignment_group", 30, "")["rows"]
worst = max([t for t in team if t["tickets"] >= 150], key=lambda t: t["breach_rate"])

inc_root_causes = [
    {"id": "RC1", "title": "Tickets bounced between teams (reassignment)", "category": "process",
     "finding": f"Breach rate rises from {r0:.0%} with no reassignment to {r1:.0%} after one and {r5:.0%} after five or more.",
     "evidence": [{"segment": "0 reassignments", "breach_rate": num(r0)},
                  {"segment": "1 reassignment", "breach_rate": num(r1)},
                  {"segment": "2+ reassignments", "breach_rate": num(r2['breached'].mean()), "tickets": len(r2)},
                  {"segment": "5+ reassignments", "breach_rate": num(r5)}],
     "affected_tickets_current_period": len(cur_r2),
     "share_of_breaches_current_period": num(cur_r2["breached"].sum() / cur["breached"].sum()),
     "confidence": "high", "linked_recommendations": ["REC1"],
     "note": "Correlation from historical data; reassignment is an outcome, so it is used for root-cause analysis, not as a model input."},
    {"id": "RC2", "title": f"{worst['value']} is under strain", "category": "capacity",
     "finding": f"{worst['value']} breached {worst['breach_rate']:.0%} of its {worst['tickets']} tickets this period vs {cur['breached'].mean():.0%} overall.",
     "evidence": [{"segment": worst["value"], "breach_rate": worst["breach_rate"], "tickets": worst["tickets"]},
                  {"segment": "all teams", "breach_rate": num(cur["breached"].mean()), "tickets": len(cur)}],
     "affected_tickets_current_period": worst["tickets"],
     "share_of_breaches_current_period": worst["share_of_all_breaches"],
     "confidence": "medium", "linked_recommendations": ["REC2"]},
    {"id": "RC3", "title": "Reopened tickets", "category": "quality",
     "finding": f"Reopened tickets breach {ro1:.0%} of the time vs {ro0:.0%} for others.",
     "evidence": [{"segment": "reopened", "breach_rate": num(ro1)}, {"segment": "not reopened", "breach_rate": num(ro0)}],
     "affected_tickets_current_period": int((cur["final_reopen_count"] > 0).sum()),
     "share_of_breaches_current_period": num(cur.loc[cur["final_reopen_count"] > 0, "breached"].sum() / cur["breached"].sum()),
     "confidence": "medium", "linked_recommendations": []},
]

# Anomalies: weeks whose breach rate is far from normal (|z| > 2 using median & MAD)
def anomalies(series, metric, label):
    med = series.median()
    mad = (series - med).abs().median() * 1.4826
    out = []
    for d, v in series.items():
        z = (v - med) / mad if mad else 0
        if abs(z) > 2:
            out.append({"week_ending": d.strftime("%Y-%m-%d"), "metric": metric, "label": label, "actual": num(v),
                        "expected": num(med), "deviation_score": num(z, 1),
                        "severity": "critical" if abs(z) > 4 else "warning",
                        "direction": "above_normal" if v > med else "below_normal"})
    return out
inc_anomalies = anomalies(weekly["breach_rate"], "breach_rate", "Weekly SLA breach rate")
print("Anomalies:", inc_anomalies)

Anomalies: [{'week_ending': '2016-03-06', 'metric': 'breach_rate', 'label': 'Weekly SLA breach rate', 'actual': 0.559, 'expected': 0.251, 'deviation_score': 4.4, 'severity': 'critical', 'direction': 'above_normal'}, {'week_ending': '2016-03-13', 'metric': 'breach_rate', 'label': 'Weekly SLA breach rate', 'actual': 0.958, 'expected': 0.251, 'deviation_score': 10.2, 'severity': 'critical', 'direction': 'above_normal'}, {'week_ending': '2016-03-20', 'metric': 'breach_rate', 'label': 'Weekly SLA breach rate', 'actual': 0.656, 'expected': 0.251, 'deviation_score': 5.8, 'severity': 'critical', 'direction': 'above_normal'}, {'week_ending': '2016-03-27', 'metric': 'breach_rate', 'label': 'Weekly SLA breach rate', 'actual': 0.392, 'expected': 0.251, 'deviation_score': 2.0, 'severity': 'warning', 'direction': 'above_normal'}]


### Forecast: tickets per team for the next 7 days
Simple **seasonal naive** method: for each weekday, the average of the same weekday over the last 4 weeks.
Capacity = average tickets the team resolved per day over the last 4 weeks. (Person 2 will replace this with a real forecasting model.)

In [6]:
last_day = inc["opened_at"].max().normalize()
recent = inc[inc["opened_at"] >= last_day - pd.Timedelta(days=28)]
top_teams = cur["assignment_group"].value_counts().head(5).index
inc_forecast = {"method": "seasonal naive (same weekday, last 4 weeks)", "horizon_days": 7, "teams": []}
for tm in top_teams:
    r = recent[recent["assignment_group"] == tm]
    daily = r.set_index("opened_at").resample("D").size()
    by_wd = daily.groupby(daily.index.dayofweek).mean()
    res = inc[(inc["assignment_group"] == tm) & (inc["resolved_at"] >= last_day - pd.Timedelta(days=28))]
    capacity = len(res) / 28
    days = []
    for k in range(1, 8):
        d = last_day + pd.Timedelta(days=k)
        f = by_wd.get(d.dayofweek, 0)
        days.append({"date": d.strftime("%Y-%m-%d"), "expected_tickets": num(f, 1),
                     "expected_capacity": num(capacity, 1), "overload": bool(f > capacity * 1.1)})
    inc_forecast["teams"].append({"team": tm, "expected_tickets_7d": num(sum(x["expected_tickets"] for x in days), 0),
                                  "expected_capacity_7d": num(capacity * 7, 0),
                                  "overloaded_days": sum(x["overload"] for x in days), "daily": days})
pd.DataFrame(inc_forecast["teams"])[["team", "expected_tickets_7d", "expected_capacity_7d", "overloaded_days"]]

,team,expected_tickets_7d,expected_capacity_7d,overloaded_days
0,Group 70,882.0,914.0,4
1,Group 20,105.0,134.0,3
2,Group 65,54.0,64.0,2
3,Group 25,45.0,56.0,3
4,Group 24,41.0,50.0,2


### Predictions with reasons (top 10 riskiest tickets in the current period)

In [7]:
INC_LABELS = {
    "recent_resolved_7d": "Tickets resolved helpdesk-wide in last 7 days",
    "assignment_group_cat": "Assigned team", "group_recent_breach_rate_7d": "Team's breach rate in last 7 days",
    "opened_by_cat": "Opened by (agent)", "u_symptom_cat": "Reported symptom", "recent_breach_rate_7d": "Helpdesk breach rate in last 7 days",
    "subcategory_cat": "Subcategory", "category_cat": "Category", "category_past_log_hours": "Category's usual resolution time (log hours)",
    "priority_n": "Priority (1 = Critical)", "open_weekday": "Day of week opened (0 = Mon)", "knowledge": "Knowledge article linked",
    "location_cat": "Location", "group_backlog_at_open": "Team backlog when opened", "category_past_breach_rate": "Category's past breach rate",
    "assignment_group_past_breach_rate": "Team's past breach rate", "group_opened_last_24h": "Tickets the team got in last 24h",
    "group_opened_last_7d": "Tickets the team got in last 7 days", "all_opened_last_24h": "Tickets helpdesk got in last 24h",
    "open_hour": "Hour opened", "impact_n": "Impact (1 = High)", "urgency_n": "Urgency (1 = High)",
    "caller_tickets_last_30d": "Caller's tickets in last 30 days", "location_past_breach_rate": "Location's past breach rate",
    "caller_id_past_breach_rate": "Caller's past breach rate",
}
X = feat.drop(columns=["number", "opened_at", "split", "breached"])
is_train = feat["split"] == "train"
m_inc, p_inc, inc_perf = train_model(X, feat["breached"], is_train)
feat["risk"] = p_inc
test = feat[~is_train]
hi, med = np.quantile(test["risk"], 0.9), np.quantile(test["risk"], 0.6)

top = test.sort_values("risk", ascending=False).head(10)
reasons = explain(m_inc, X.loc[top.index], X[is_train], INC_LABELS)
ctx = inc.set_index("number")
train_hist = inc[inc["split"] == "train"]
inc_predictions = []
for (idx, row), rs in zip(top.iterrows(), reasons):
    c = ctx.loc[row["number"]]
    sim = train_hist[(train_hist["category"] == c["category"]) & (train_hist["assignment_group"] == c["assignment_group"])]
    pct = (test["risk"] < row["risk"]).mean()
    inc_predictions.append({
        "id": row["number"], "item_type": "ticket",
        "context": {"opened_at": c["opened_at"].strftime("%Y-%m-%dT%H:%M"), "team": c["assignment_group"],
                    "category": c["category"], "subcategory": c["subcategory"], "priority": int(c["priority_n"]),
                    "location": c["location"], "contact_type": c["contact_type"]},
        "prediction": {"target": "sla_breach", "risk": num(row["risk"]), "risk_level": risk_level(row["risk"], hi, med),
                       "risk_percentile": num(pct * 100, 1), "prediction_moment": "ticket opened"},
        "top_reasons": rs,
        "explanation_summary": f"{risk_level(row['risk'], hi, med).capitalize()} risk ({row['risk']:.0%}) of missing SLA, mainly because: "
                               + "; ".join(r["sentence"] for r in rs if r["direction"] == "increases risk") + ".",
        "similar_past_cases": {"rule": "same category and team, training period", "count": len(sim),
                               "breach_rate": num(sim["breached"].mean()) if len(sim) else None,
                               "median_resolution_hours": num(sim["hours_to_resolve"].median(), 1) if len(sim) else None},
        "recommended_action": {"recommendation_id": "REC3", "action": "Fast-track: senior agent picks it up within 1 hour",
                               "urgency": "now" if row["risk"] >= hi else "today"},
        "actual_outcome": {"breached": bool(row["breached"]), "note": "known only because this is historical data; hidden in a live system"},
    })
print("Model:", inc_perf)
pd.DataFrame([{"id": p["id"], "risk": p["prediction"]["risk"], "team": p["context"]["team"],
               "reason_1": p["top_reasons"][0]["sentence"]} for p in inc_predictions])

Model: {'algorithm': 'LightGBM (gradient boosted trees)', 'test_auc': 0.771, 'train_auc': 0.92, 'base_rate_test': 0.225, 'precision_top_10pct': 0.517, 'lift_top_10pct': 2.3, 'n_features': 33, 'train_rows': 19715, 'test_rows': 4929}


,id,risk,team,reason_1
0,INC0034301,0.969,Group 9,Tickets resolved helpdesk-wide in last 7 days:...
1,INC0034330,0.968,Group 9,Tickets resolved helpdesk-wide in last 7 days:...
2,INC0034268,0.963,Group 9,Tickets resolved helpdesk-wide in last 7 days:...
3,INC0034416,0.960,Group 22,Tickets resolved helpdesk-wide in last 7 days:...
4,INC0034423,0.959,Group 22,Tickets resolved helpdesk-wide in last 7 days:...
5,INC0033653,0.957,Group 53,Tickets resolved helpdesk-wide in last 7 days:...
6,INC0034429,0.949,Group 57,Tickets resolved helpdesk-wide in last 7 days:...
7,INC0034048,0.948,Group 9,Tickets resolved helpdesk-wide in last 7 days:...
8,INC0034380,0.947,Group 55,Tickets resolved helpdesk-wide in last 7 days:...
9,INC0033915,0.946,Group 66,Tickets resolved helpdesk-wide in last 7 days:...


### Recommendations and impact simulation
Expected impacts use a **simple historical estimate** (stated in `estimate_method`). The real simulator (Person 3) will replace these later.

In [8]:
base_breaches = int(cur["breached"].sum())
base_rate = cur["breached"].mean()

# REC1: route right first time -> tickets with 2+ reassignments behave like 1-reassignment tickets
saved1 = cur_r2["breached"].sum() - len(cur_r2) * r1
# REC2: bring worst team to overall rate (via rebalancing)
saved2 = worst["breaches"] - worst["tickets"] * base_rate
# REC3: fast-track top 10% risk tickets; assume fast-tracking prevents 30% of their breaches (assumption, to validate)
cur_feat = feat[(feat["opened_at"] >= cutoff)]
top10 = cur_feat[cur_feat["risk"] >= hi]
saved3 = top10["breached"].sum() * 0.30

def rec(id, title, rtype, desc, target, evidence, saved, method, effort, owner, conf, assumptions):
    new_rate = (base_breaches - saved) / len(cur)
    return {"id": id, "title": title, "type": rtype, "description": desc, "target": target, "evidence": evidence,
            "expected_impact": {"metric": "sla_breach_rate", "baseline": num(base_rate), "expected": num(new_rate),
                                "change_pct_points": num((new_rate - base_rate) * 100, 1),
                                "breaches_avoided_per_period": num(saved, 0),
                                "range_breaches_avoided": [num(saved * 0.5, 0), num(saved, 0)]},
            "estimate_method": method, "assumptions": assumptions, "confidence": conf, "effort": effort,
            "suggested_owner": owner, "linked_root_causes": [r["id"] for r in inc_root_causes if id in r["linked_recommendations"]],
            "status": "pending", "approval": {"decided_by": None, "decided_at": None, "decision": None, "comment": None}}

inc_recs = [
    rec("REC1", "Route tickets to the right team first time", "process",
        "Use the model's category/symptom signals to suggest the correct team at opening; require a reason when reassigning.",
        {"tickets": "all new tickets", "focus": "categories with most reassignments"},
        ["RC1: breach rate climbs with every reassignment"], saved1,
        "Tickets with 2+ reassignments in the current period assumed to breach at the 1-reassignment rate.",
        "medium", "Helpdesk lead", "medium", ["Better routing cuts repeat reassignments, not the first one"]),
    rec("REC2", f"Rebalance workload away from {worst['value']}", "capacity",
        f"Move part of {worst['value']}'s incoming tickets to teams with spare capacity, or add temporary staff.",
        {"team": worst["value"]}, ["RC2: team breach rate far above average"], saved2,
        "Team assumed to reach the overall breach rate after rebalancing.", "low", "Operations manager", "low",
        ["Receiving teams have the skills and spare capacity"]),
    rec("REC3", "Fast-track the 10% highest-risk tickets", "prioritisation",
        "Every new ticket in the model's top 10% risk goes to a senior agent within 1 hour.",
        {"tickets": f"risk ≥ {hi:.2f} (top 10%)", "count_current_period": len(top10)},
        [f"Top-10% tickets breach {top10['breached'].mean():.0%} vs {base_rate:.0%} overall"], saved3,
        "Assumes fast-tracking prevents 30% of breaches among flagged tickets (to be validated in a pilot).",
        "low", "Shift supervisor", "medium", ["30% prevention rate is an assumption"]),
]
inc_impact = {"period": "current", "baseline": {"tickets": len(cur), "breaches": base_breaches, "breach_rate": num(base_rate)},
              "scenarios": [{"recommendation_id": r["id"], "breach_rate": r["expected_impact"]["expected"],
                             "breaches_avoided": r["expected_impact"]["breaches_avoided_per_period"]} for r in inc_recs],
              "note": "Scenarios are estimated separately and are not additive (they overlap)."}
pd.DataFrame([{"id": r["id"], "title": r["title"], **r["expected_impact"]} for r in inc_recs])

,id,title,metric,baseline,expected,change_pct_points,breaches_avoided_per_period,range_breaches_avoided
0,REC1,Route tickets to the right team first time,sla_breach_rate,0.225,0.202,-2.2,111.0,"[55.0, 111.0]"
1,REC2,Rebalance workload away from Group 20,sla_breach_rate,0.225,0.214,-1.1,53.0,"[27.0, 53.0]"
2,REC3,Fast-track the 10% highest-risk tickets,sla_breach_rate,0.225,0.209,-1.6,76.0,"[38.0, 76.0]"


### Assemble and save the incidents JSON

In [9]:
inc_json = {
    "schema_version": "1.0",
    "meta": {"analysis_id": "inc-" + GENERATED_AT[:10], "generated_at": GENERATED_AT, "domain": "support_tickets",
             "dataset_name": "UCI Incident Management Process Enriched Event Log", "item_type": "ticket",
             "target": {"name": "sla_breach", "definition": "ticket missed its SLA deadline (made_sla = False on the last event)"},
             "prediction_moment": "when the ticket is opened",
             "data_period": {"start": inc["opened_at"].min().strftime("%Y-%m-%d"), "end": inc["opened_at"].max().strftime("%Y-%m-%d")},
             "current_period": {"start": cutoff.strftime("%Y-%m-%d"), "end": inc["opened_at"].max().strftime("%Y-%m-%d")},
             "previous_period": {"start": prev_start.strftime("%Y-%m-%d"), "end": (cutoff - pd.Timedelta(minutes=1)).strftime("%Y-%m-%d")},
             "currency": None, "timezone": "local time of source system"},
    "data_quality": dq_section(REPORTS / "incidents_quality_report.csv", 141712, len(inc), "tickets",
        warnings=[{"level": "info", "message": "Missing values were stored as '?' and converted to empty."},
                  {"level": "warning", "message": "Breach rate drifts strongly over time (mid-March spike); compare periods with care."},
                  {"level": "info", "message": "Five columns (caused_by, vendor, cmdb_ci, rfc, problem_id) are 98-100% empty and were reduced to yes/no flags."}],
        modules=[{"module": "kpis", "enabled": True}, {"module": "root_cause", "enabled": True},
                 {"module": "sla_risk_model", "enabled": True}, {"module": "forecast", "enabled": True},
                 {"module": "revenue_impact", "enabled": False, "reason": "dataset has no cost or revenue column"}]),
    "kpis": inc_kpi_list, "trends": inc_trends, "breakdowns": inc_breakdowns, "root_causes": inc_root_causes,
    "anomalies": inc_anomalies, "forecast": inc_forecast,
    "predictions": {"count_shown": len(inc_predictions), "sorted_by": "risk", "risk_level_rules":
                    {"high": f"risk ≥ {hi:.3f} (top 10%)", "medium": f"risk ≥ {med:.3f} (top 40%)", "low": "below medium"},
                    "items": inc_predictions},
    "recommendations": inc_recs, "impact_simulation": inc_impact,
    "model_performance": {**inc_perf, "validation": "time-based split: trained on older tickets, tested on newest 20%",
                          "test_period": {"start": cutoff.strftime("%Y-%m-%d"), "end": inc["opened_at"].max().strftime("%Y-%m-%d")}},
}
(CONTRACTS / "example_incidents.json").write_text(json.dumps(inc_json, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved contracts/example_incidents.json", round((CONTRACTS / "example_incidents.json").stat().st_size / 1024), "KB")

Saved contracts/example_incidents.json 56 KB


---
# Part 2: Sales

In [10]:
so = pd.read_parquet(INTERIM / "sales_orders_clean.parquet")
sf = pd.read_parquet(PROCESSED / "sales_orders_features.parquet")
d = so[so["is_model_row"] == 1].merge(sf[["order_id", "split"]], on="order_id")
d["late"] = d["late"].astype(int)
d["delivery_days"] = (d["delivered_at"] - d["purchased_at"]).dt.total_seconds() / 86400
d["dispatch_days"] = ((d["carrier_at"] - d["approved_at"]).dt.total_seconds() / 86400).clip(lower=0)
d["revenue"] = d["items_price"] + d["items_freight"]

s_cut = sf.loc[sf["split"] == "test", "purchased_at"].min()
s_len = sf["purchased_at"].max() - s_cut
s_prev_start = s_cut - s_len
scur = d[d["purchased_at"] >= s_cut]
sprev = d[(d["purchased_at"] >= s_prev_start) & (d["purchased_at"] < s_cut)]
print("Current:", len(scur), "orders   Previous:", len(sprev))

def s_kpis(df):
    return {"orders": len(df), "late": df["late"].mean(), "days": df["delivery_days"].mean(),
            "review": df["review_score"].mean(), "revenue": df["revenue"].sum(), "dispatch": df["dispatch_days"].mean()}
a, b = s_kpis(scur), s_kpis(sprev)
sales_kpis = [
    kpi("orders_delivered", "Orders delivered", a["orders"], b["orders"], "orders", "Delivered orders purchased in the period", None, d=0),
    kpi("revenue", "Revenue (items + freight)", a["revenue"], b["revenue"], "BRL", "Sum of item price and freight", True, 5, 20, d=0, mode="change"),
    kpi("late_delivery_rate", "Late delivery rate", a["late"], b["late"], "ratio", "Share delivered after the promised date", False, 0.07, 0.10),
    kpi("avg_delivery_days", "Average delivery time", a["days"], b["days"], "days", "Days from purchase to delivery", False, 12, 15, d=1),
    kpi("avg_dispatch_days", "Average seller dispatch time", a["dispatch"], b["dispatch"], "days", "Days from approval to carrier pickup", False, 3, 4, d=2),
    kpi("avg_review_score", "Average review score", a["review"], b["review"], "stars (1-5)", "Mean review score of delivered orders", True, 4.0, 3.5, d=2),
]

w = d.set_index("purchased_at").resample("W-SUN")
sw = pd.DataFrame({"orders": w.size(), "late_rate": w["late"].mean(), "revenue": w["revenue"].sum(),
                   "avg_delivery_days": w["delivery_days"].mean()}).dropna()
sales_trends = {"granularity": "week", "series": [
    {"week_ending": i.strftime("%Y-%m-%d"), "orders": int(r.orders), "late_rate": num(r.late_rate),
     "revenue": num(r.revenue, 0), "avg_delivery_days": num(r.avg_delivery_days, 1)} for i, r in sw.iterrows()]}

def s_breakdown(df, col, top, label, min_n=100):
    total_late = df["late"].sum()
    overall = df["late"].mean()
    g = df.groupby(col).agg(orders=("order_id", "size"), late_rate=("late", "mean"), late_orders=("late", "sum"),
                            avg_delivery_days=("delivery_days", "mean"), avg_review=("review_score", "mean"),
                            revenue=("revenue", "sum"))
    g = g[g["orders"] >= min_n].sort_values("late_orders", ascending=False).head(top)
    return {"dimension": col, "label": label, "sorted_by": "late_orders", "rows": [
        {"value": str(i), "orders": int(r.orders), "late_rate": num(r.late_rate), "late_orders": int(r.late_orders),
         "share_of_all_late": num(r.late_orders / total_late), "vs_overall_late_rate": num(r.late_rate - overall),
         "avg_delivery_days": num(r.avg_delivery_days, 1), "avg_review_score": num(r.avg_review, 2), "revenue": num(r.revenue, 0),
         "flag": "above_average" if r.late_rate > overall * 1.2 else "normal"} for i, r in g.iterrows()]}

sales_breakdowns = [s_breakdown(scur, "customer_state", 8, "Customer state"),
                    s_breakdown(scur, "main_category", 8, "Product category"),
                    s_breakdown(scur, "main_seller_id", 8, "Seller", min_n=50)]

# Root causes
q = d["dispatch_days"].quantile([0.5, 0.9])
fast, slow = d[d["dispatch_days"] <= q[0.5]], d[d["dispatch_days"] >= q[0.9]]
d["promise_bucket"] = pd.qcut(d["promised_days"] if "promised_days" in d else (d["estimated_at"] - d["approved_at"]).dt.days, 4, labels=False)
promised = (d["estimated_at"] - d["approved_at"]).dt.total_seconds() / 86400
short, long_ = d[promised <= promised.quantile(0.25)], d[promised >= promised.quantile(0.75)]
northeast = {"AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"}
ne, sp = d[d["customer_state"].isin(northeast)], d[d["customer_state"] == "SP"]
on_time_rev, late_rev = d.loc[d["late"] == 0, "review_score"].mean(), d.loc[d["late"] == 1, "review_score"].mean()
scur_slow = scur[scur["dispatch_days"] >= q[0.9]]

sales_root_causes = [
    {"id": "RC1", "title": "Slow seller dispatch", "category": "supplier",
     "finding": f"Orders the seller took {q[0.9]:.1f}+ days to ship are late {slow['late'].mean():.0%} of the time vs {fast['late'].mean():.0%} when shipped within {q[0.5]:.1f} days.",
     "evidence": [{"segment": f"dispatch ≤ {q[0.5]:.1f} days", "late_rate": num(fast["late"].mean()), "orders": len(fast)},
                  {"segment": f"dispatch ≥ {q[0.9]:.1f} days", "late_rate": num(slow["late"].mean()), "orders": len(slow)}],
     "affected_orders_current_period": len(scur_slow),
     "share_of_late_current_period": num(scur_slow["late"].sum() / scur["late"].sum()),
     "confidence": "high", "linked_recommendations": ["REC1"]},
    {"id": "RC2", "title": "Long-distance deliveries to the North-East", "category": "logistics",
     "finding": f"North-East customers get late orders {ne['late'].mean():.0%} of the time vs {sp['late'].mean():.0%} in São Paulo.",
     "evidence": [{"segment": "North-East states", "late_rate": num(ne["late"].mean()), "orders": len(ne)},
                  {"segment": "São Paulo (SP)", "late_rate": num(sp["late"].mean()), "orders": len(sp)}],
     "affected_orders_current_period": int(scur["customer_state"].isin(northeast).sum()),
     "share_of_late_current_period": num(scur.loc[scur["customer_state"].isin(northeast), "late"].sum() / scur["late"].sum()),
     "confidence": "high", "linked_recommendations": ["REC2"]},
    {"id": "RC3", "title": "Tight delivery promises", "category": "policy",
     "finding": f"Orders with the shortest 25% of promises are late {short['late'].mean():.0%} vs {long_['late'].mean():.0%} for the longest 25%.",
     "evidence": [{"segment": "shortest 25% promises", "late_rate": num(short["late"].mean())},
                  {"segment": "longest 25% promises", "late_rate": num(long_["late"].mean())}],
     "affected_orders_current_period": None, "share_of_late_current_period": None,
     "confidence": "medium", "linked_recommendations": ["REC2"]},
    {"id": "RC4", "title": "Business impact: late orders damage reviews", "category": "customer",
     "finding": f"Late orders average {late_rev:.1f} stars vs {on_time_rev:.1f} on time.",
     "evidence": [{"segment": "on time", "avg_review_score": num(on_time_rev, 2)}, {"segment": "late", "avg_review_score": num(late_rev, 2)}],
     "affected_orders_current_period": int(scur["late"].sum()), "share_of_late_current_period": 1.0,
     "confidence": "high", "linked_recommendations": []},
]
sales_anomalies = anomalies(sw["late_rate"], "late_rate", "Weekly late delivery rate")

# Forecast: platform orders per day next 7 days
last = d["purchased_at"].max().normalize()
rd = d[d["purchased_at"] >= last - pd.Timedelta(days=28)].set_index("purchased_at").resample("D").size()
by_wd = rd.groupby(rd.index.dayofweek).mean()
sales_forecast = {"method": "seasonal naive (same weekday, last 4 weeks)", "horizon_days": 7,
                  "note": "Recent weeks are under-counted because slow orders hadn't been delivered when data was exported.",
                  "daily": [{"date": (last + pd.Timedelta(days=k)).strftime("%Y-%m-%d"),
                             "expected_orders": num(by_wd.get((last + pd.Timedelta(days=k)).dayofweek, 0), 0)} for k in range(1, 8)]}

# Predictions
S_LABELS = {
    "customer_state_cat": "Customer state", "promised_days": "Days promised for delivery", "purchase_month": "Purchase month",
    "customer_state_past_late_rate": "Customer state's past late rate", "main_category_cat": "Product category",
    "distance_km": "Seller-to-customer distance (km)", "seller_past_dispatch_days": "Seller's usual dispatch time (days)",
    "is_holiday_season": "Holiday season", "route_past_late_rate": "Route's past late rate", "seller_state_cat": "Seller state",
    "items_freight": "Freight charged (BRL)", "category_past_late_rate": "Category's past late rate", "same_state": "Same state",
    "seller_past_late_rate": "Seller's past late rate", "days_to_ship_limit": "Days seller has to ship",
    "approval_lag_hours": "Hours from purchase to approval", "seller_orders_last_30d": "Seller's orders in last 30 days",
    "seller_orders_last_7d": "Seller's orders in last 7 days", "total_weight_g": "Total weight (g)", "items_price": "Item price (BRL)",
    "freight_ratio": "Freight / price", "max_installments": "Payment instalments", "approve_weekday": "Day of week approved (0 = Mon)",
}
Xs = sf.drop(columns=["order_id", "purchased_at", "approved_at", "split", "late"])
s_train = sf["split"] == "train"
m_s, p_s, sales_perf = train_model(Xs, sf["late"], s_train)
sf["risk"] = p_s
stest = sf[~s_train]
shi, smed = np.quantile(stest["risk"], 0.9), np.quantile(stest["risk"], 0.6)
stop = stest.sort_values("risk", ascending=False).head(10)
sreasons = explain(m_s, Xs.loc[stop.index], Xs[s_train], S_LABELS)
sctx = d.set_index("order_id")
strain_hist = d[d["split"] == "train"]
sales_predictions = []
for (idx, row), rs in zip(stop.iterrows(), sreasons):
    c = sctx.loc[row["order_id"]]
    sim = strain_hist[(strain_hist["seller_state"] == c["seller_state"]) & (strain_hist["customer_state"] == c["customer_state"])]
    sales_predictions.append({
        "id": row["order_id"], "item_type": "order",
        "context": {"approved_at": c["approved_at"].strftime("%Y-%m-%dT%H:%M"), "promised_by": c["estimated_at"].strftime("%Y-%m-%d"),
                    "customer_state": c["customer_state"], "customer_city": c["customer_city"], "seller_id": c["main_seller_id"],
                    "seller_state": c["seller_state"], "category": c["main_category"], "order_value": num(c["revenue"], 2),
                    "n_items": int(c["n_items"]), "payment_type": c["main_payment_type"]},
        "prediction": {"target": "late_delivery", "risk": num(row["risk"]), "risk_level": risk_level(row["risk"], shi, smed),
                       "risk_percentile": num((stest["risk"] < row["risk"]).mean() * 100, 1), "prediction_moment": "order approved"},
        "top_reasons": rs,
        "explanation_summary": f"{risk_level(row['risk'], shi, smed).capitalize()} risk ({row['risk']:.0%}) of late delivery, mainly because: "
                               + "; ".join(r["sentence"] for r in rs if r["direction"] == "increases risk") + ".",
        "similar_past_cases": {"rule": "same seller state → customer state route, training period", "count": len(sim),
                               "late_rate": num(sim["late"].mean()) if len(sim) else None,
                               "avg_delivery_days": num(sim["delivery_days"].mean(), 1) if len(sim) else None},
        "recommended_action": {"recommendation_id": "REC3", "action": "Contact seller to ship within 24h and use express carrier",
                               "urgency": "now" if row["risk"] >= shi else "today"},
        "actual_outcome": {"late": bool(row["late"]), "note": "known only because this is historical data; hidden in a live system"},
    })

# Recommendations
sbase_late, sbase_rate = int(scur["late"].sum()), scur["late"].mean()
saved1 = scur_slow["late"].sum() - len(scur_slow) * d.loc[d["dispatch_days"].between(q[0.5], q[0.9]), "late"].mean()
cur_ne = scur[scur["customer_state"].isin(northeast)]
saved2 = cur_ne["late"].sum() * 0.40
stop10 = sf[(sf["purchased_at"] >= s_cut) & (sf["risk"] >= shi)]
saved3 = stop10["late"].sum() * 0.30

def srec(id, title, rtype, desc, target, evidence, saved, method, effort, owner, conf, assumptions):
    new_rate = (sbase_late - saved) / len(scur)
    return {"id": id, "title": title, "type": rtype, "description": desc, "target": target, "evidence": evidence,
            "expected_impact": {"metric": "late_delivery_rate", "baseline": num(sbase_rate), "expected": num(new_rate),
                                "change_pct_points": num((new_rate - sbase_rate) * 100, 1),
                                "late_orders_avoided_per_period": num(saved, 0),
                                "range_late_orders_avoided": [num(saved * 0.5, 0), num(saved, 0)],
                                "review_score_gain_estimate": num(saved * (on_time_rev - late_rev) / len(scur), 3)},
            "estimate_method": method, "assumptions": assumptions, "confidence": conf, "effort": effort, "suggested_owner": owner,
            "linked_root_causes": [r["id"] for r in sales_root_causes if id in r["linked_recommendations"]],
            "status": "pending", "approval": {"decided_by": None, "decided_at": None, "decision": None, "comment": None}}

sales_recs = [
    srec("REC1", "Enforce a dispatch deadline for slow sellers", "supplier",
         "Alert sellers whose dispatch time is in the slowest 10%; escalate after 2 days; deprioritise repeat offenders in search.",
         {"sellers": "slowest 10% by dispatch time"}, ["RC1"], saved1,
         "Slow-dispatch orders assumed to be late at the rate of medium-dispatch orders.", "medium", "Seller operations", "medium",
         ["Sellers respond to alerts"]),
    srec("REC2", "Add buffer days to promises for North-East routes", "policy",
         "Add 2-3 days to the promised date for high-risk routes so promises match reality.",
         {"customer_states": sorted(northeast)}, ["RC2", "RC3"], saved2,
         "Assumes 40% of North-East late orders arrive within the added buffer (validate with delivery-day distribution).",
         "low", "Logistics manager", "medium", ["Slightly longer promises don't reduce conversion much"]),
    srec("REC3", "Proactively handle the 10% highest-risk orders", "prioritisation",
         "For each order in the model's top 10%: nudge the seller, upgrade carrier, and send the customer a delivery update.",
         {"orders": f"risk ≥ {shi:.2f} (top 10%)", "count_current_period": len(stop10)},
         [f"Top-10% orders are late {stop10['late'].mean():.0%} vs {sbase_rate:.0%} overall"], saved3,
         "Assumes proactive handling prevents 30% of late deliveries among flagged orders (to be validated).",
         "medium", "Customer experience team", "medium", ["30% prevention rate is an assumption"]),
]
sales_impact = {"period": "current", "baseline": {"orders": len(scur), "late_orders": sbase_late, "late_rate": num(sbase_rate),
                "avg_review_score": num(scur["review_score"].mean(), 2)},
                "scenarios": [{"recommendation_id": r["id"], "late_rate": r["expected_impact"]["expected"],
                               "late_orders_avoided": r["expected_impact"]["late_orders_avoided_per_period"]} for r in sales_recs],
                "note": "Scenarios are estimated separately and are not additive (they overlap)."}

sales_json = {
    "schema_version": "1.0",
    "meta": {"analysis_id": "sales-" + GENERATED_AT[:10], "generated_at": GENERATED_AT, "domain": "sales_orders",
             "dataset_name": "Olist Brazilian E-Commerce", "item_type": "order",
             "target": {"name": "late_delivery", "definition": "delivered after the estimated (promised) delivery date"},
             "prediction_moment": "when the order is approved",
             "data_period": {"start": d["purchased_at"].min().strftime("%Y-%m-%d"), "end": d["purchased_at"].max().strftime("%Y-%m-%d")},
             "current_period": {"start": s_cut.strftime("%Y-%m-%d"), "end": d["purchased_at"].max().strftime("%Y-%m-%d")},
             "previous_period": {"start": s_prev_start.strftime("%Y-%m-%d"), "end": (s_cut - pd.Timedelta(minutes=1)).strftime("%Y-%m-%d")},
             "currency": "BRL", "timezone": "America/Sao_Paulo"},
    "data_quality": dq_section(REPORTS / "sales_quality_report.csv", 99441, len(d), "orders",
        warnings=[{"level": "warning", "message": "Recent orders are right-censored: slow orders from late Aug 2018 had not arrived at export time, so the current late rate is understated."},
                  {"level": "info", "message": "275 customer and 216 seller locations could not be geocoded; distance is empty for those orders."},
                  {"level": "info", "message": "Only delivered orders are used for late-delivery analysis; 2,149 other orders are kept for status analysis."}],
        modules=[{"module": "kpis", "enabled": True}, {"module": "root_cause", "enabled": True},
                 {"module": "late_risk_model", "enabled": True}, {"module": "forecast", "enabled": True},
                 {"module": "revenue_impact", "enabled": True}]),
    "kpis": sales_kpis, "trends": sales_trends, "breakdowns": sales_breakdowns, "root_causes": sales_root_causes,
    "anomalies": sales_anomalies, "forecast": sales_forecast,
    "predictions": {"count_shown": len(sales_predictions), "sorted_by": "risk", "risk_level_rules":
                    {"high": f"risk ≥ {shi:.3f} (top 10%)", "medium": f"risk ≥ {smed:.3f} (top 40%)", "low": "below medium"},
                    "items": sales_predictions},
    "recommendations": sales_recs, "impact_simulation": sales_impact,
    "model_performance": {**sales_perf, "validation": "time-based split: trained on older orders, tested on newest 20%",
                          "test_period": {"start": s_cut.strftime("%Y-%m-%d"), "end": d["purchased_at"].max().strftime("%Y-%m-%d")}},
}
(CONTRACTS / "example_sales.json").write_text(json.dumps(sales_json, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved contracts/example_sales.json", round((CONTRACTS / "example_sales.json").stat().st_size / 1024), "KB")
pd.DataFrame(sales_kpis)[["label", "value", "previous_value", "change_pct", "trend", "status"]]

Current: 19204 orders   Previous: 22105
Saved contracts/example_sales.json 68 KB


,label,value,previous_value,change_pct,trend,status
0,Orders delivered,19204.000,22105.000,-13.1,down,info
1,Revenue (items + freight),3141322.000,3592162.000,-12.6,worsening,warning
2,Late delivery rate,0.053,0.135,-60.6,improving,good
3,Average delivery time,8.800,13.700,-36.0,improving,good
4,Average seller dispatch time,2.290,2.560,-10.5,improving,good
5,Average review score,4.310,4.030,6.9,improving,good


## Check: both files load back as valid JSON

In [11]:
for f in ["example_incidents.json", "example_sales.json"]:
    obj = json.loads((CONTRACTS / f).read_text(encoding="utf-8"))
    print(f, "→ sections:", list(obj.keys()))

example_incidents.json → sections: ['schema_version', 'meta', 'data_quality', 'kpis', 'trends', 'breakdowns', 'root_causes', 'anomalies', 'forecast', 'predictions', 'recommendations', 'impact_simulation', 'model_performance']
example_sales.json → sections: ['schema_version', 'meta', 'data_quality', 'kpis', 'trends', 'breakdowns', 'root_causes', 'anomalies', 'forecast', 'predictions', 'recommendations', 'impact_simulation', 'model_performance']
